In [0]:
# Import necessary PySpark libraries for data ingestion
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
catalog_name = 'ecommerce'

BRAND

In [0]:

brand_name = StructType() \
  .add('brand_code', StringType(), False) \
  .add('brand_name', StringType(), True) \
  .add('category_code', StringType(), False)

In [0]:
%skip
#checking metadata columns of a table
df_brands.select("_metadata").printSchema()

# Display all _metadata fields as individual columns alongside the table columns
# `"*"` selects all regular columns, `col("_metadata.*")` expands the metadata struct

display(df_brands.select("*", col("_metadata.*")))

In [0]:
raw_data_path ='/Volumes/ecommerce/source_data/raw/brands/*.csv'
df_brands = spark.read.option('header', True).option('delimiter', ',').schema(brand_name).csv(raw_data_path)
df_brands = df_brands.withColumn('ingested_at', current_timestamp())
df_brands = df_brands.withColumn('_source_file', col("_metadata.file_path"))
display(df_brands)

In [0]:
df_brands.write.format("delta").mode("overwrite").option("mergeSchema", True).saveAsTable(f"{catalog_name}.bronze.brz_brand")

CATEGORY

In [0]:
Category_name = StructType() \
  .add('category_code', StringType(), False) \
  .add('category_name', StringType(), True)

df = spark.read.option("header", True).option("Schema", "category_name").csv("/Volumes/ecommerce/source_data/raw/category/*.csv")
display(df)

df = df.withColumn('ingested_at', current_timestamp())
df = df.withColumn('_source_file', col("_metadata.file_path"))
display(df)


df.write.format("delta").mode("overwrite").option("mergeSchema", True).saveAsTable(f"{catalog_name}.bronze.brz_category")


CUSTOMERS

In [0]:
customer_name = StructType() \
  .add('customer_id', StringType(), False) \
  .add('phone', StringType(), True)\
  .add('country_code', StringType(), True)\
  .add('country', StringType(), True)\
  .add('state', StringType(), True)


df = spark.read.option("header", True).option("Schema", "customer_name").csv("/Volumes/ecommerce/source_data/raw/customers/*.csv")
display(df)
df = df.withColumn('ingested_at', current_timestamp())
df = df.withColumn('_source_file', col("_metadata.file_path"))
display(df)


df.write.format("delta").mode("overwrite").option("mergeSchema", True).saveAsTable(f"{catalog_name}.bronze.brz_customer")


DATE

In [0]:
date_name = StructType() \
  .add('date', StringType(), True) \
  .add('year', StringType(), True)\
  .add('day_name', StringType(), True)\
  .add('quarter', StringType(), True)\
  .add('week_of_year', StringType(), True)


df = spark.read.option("header", True).option("Schema", "customer_name").csv("/Volumes/ecommerce/source_data/raw/date/*.csv")
display(df)
df = df.withColumn('ingested_at', current_timestamp())
df = df.withColumn('_source_file', col("_metadata.file_path"))
display(df)


df.write.format("delta").mode("overwrite").option("mergeSchema", True).saveAsTable(f"{catalog_name}.bronze.brz_date")


PRODUCTS

In [0]:
products_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("sku", StringType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand_code", StringType(), True),
    StructField("color", StringType(), True),
    StructField("size", StringType(), True),
    StructField("material", StringType(), True),
    StructField("weight_grams", StringType(), True),  #datatype is string due to incoming data contain anamolies
    StructField("length_cm", StringType(), True),     #datatype is string due to incoming data contain anamolies
    StructField("width_cm", FloatType(), True),
    StructField("height_cm", FloatType(), True),
    StructField("rating_count", IntegerType(), True),
    StructField("file_name", StringType(), False),
    StructField("ingest_timestamp", TimestampType(), False)
])

# Load data using the schema defined
raw_data_path = "/Volumes/ecommerce/source_data/raw/products/*.csv"

df = spark.read.option("header", "true").option("delimiter", ",").schema(products_schema).csv(raw_data_path) \
    .withColumn("file_name", col("_metadata.file_path")) \
    .withColumn("ingest_timestamp", current_timestamp())

# Write raw data to the Bronze layer (catalog: ecommerce, schema: bronze, table: brz_products)
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_products")    